# ResiliCity ML: heat regression calibrated on satellite data
**Goal:** learn how land-cover mix predicts land-surface temperature (LST) around Chengalpattu and Chennai, so the heat score is trained on real data instead of hand-set factors.

Data: Landsat 8/9 LST (Mar-May median, 2021-2025), Sentinel-2 indices, ESA WorldCover fractions. Evaluation uses **spatial** cross-validation (test areas are never next to training areas).

Requirements: a Google Cloud project with the Earth Engine API enabled (free for non-commercial use). Run cells top to bottom.

In [ ]:
import ee
PROJECT = 'your-gcp-project-id'   # <-- change this
ee.Authenticate()                 # opens a login link, paste the code back
ee.Initialize(project=PROJECT)

In [ ]:
roi = ee.Geometry.Rectangle([79.90, 12.60, 80.35, 13.20])   # Chengalpattu to Chennai
START, END = '2021-01-01', '2026-01-01'
PRE_MONSOON = ee.Filter.calendarRange(3, 5, 'month')        # hottest, mostly clear-sky months

def mask_landsat(img):
    qa = img.select('QA_PIXEL')
    bad = qa.bitwiseAnd(0b11110).neq(0)    # dilated cloud, cirrus, cloud, shadow
    return img.updateMask(bad.Not())

def to_celsius(img):
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return lst.rename('LST_C')

l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
lst = (l8.merge(l9).filterBounds(roi).filterDate(START, END).filter(PRE_MONSOON)
         .map(mask_landsat).map(to_celsius).median())

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(roi)
        .filterDate(START, END).filter(PRE_MONSOON)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)).median())
ndvi  = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi  = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')
mndwi = s2.normalizedDifference(['B3', 'B11']).rename('MNDWI')

wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
CLASSES = {'tree': 10, 'shrub': 20, 'grass': 30, 'crop': 40, 'built': 50, 'bare': 60, 'water': 80}
frac = ee.Image.cat([wc.eq(v).rename('f_' + k) for k, v in CLASSES.items()])

stack = lst.addBands([ndvi, ndbi, mndwi, frac])

def add_xy(f):
    c = f.geometry().coordinates()
    return f.set({'lon': c.get(0), 'lat': c.get(1)})

pts = ee.FeatureCollection.randomPoints(region=roi, points=4000, seed=42).map(add_xy)
samples = stack.reduceRegions(collection=pts.map(lambda f: f.buffer(45)),   # 3x3 Landsat pixels
                              reducer=ee.Reducer.mean(), scale=30)
cols = ['LST_C', 'NDVI', 'NDBI', 'MNDWI'] + ['f_' + k for k in CLASSES] + ['lon', 'lat']
task = ee.batch.Export.table.toDrive(collection=samples.select(cols, None, False),
                                     description='resilicity_samples', folder='resilicity', fileFormat='CSV')
task.start()

In [ ]:
import time
while task.active():
    print('export running...', task.status()['state']); time.sleep(20)
print('final state:', task.status()['state'])   # should be COMPLETED

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
df = pd.read_csv('/content/drive/MyDrive/resilicity/resilicity_samples.csv')
print(df.shape)
df.head()

In [ ]:
FRAC = ['f_tree', 'f_shrub', 'f_grass', 'f_crop', 'f_built', 'f_bare', 'f_water']
IDX  = ['NDVI', 'NDBI', 'MNDWI']
df = df.dropna(subset=['LST_C'] + FRAC + IDX)
df = df[(df.f_water < 0.5) & df.LST_C.between(20, 65)].reset_index(drop=True)   # water LST is a different regime
df['block'] = (np.floor(df.lat / 0.05).astype(int).astype(str) + '_' + np.floor(df.lon / 0.05).astype(int).astype(str))
print(len(df), 'samples,', df.block.nunique(), 'spatial blocks (~5 km)')
print(df.LST_C.describe().round(1))

In [ ]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Physics prior: more built area / bare ground never cools, more vegetation / water never warms.
MONO = {'f_tree': -1, 'f_shrub': -1, 'f_grass': -1, 'f_built': 1, 'f_water': -1, 'NDVI': -1, 'NDBI': 1, 'MNDWI': -1}

def ridge(cols=None):  return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
def gboost(cols=None):
    cst = [MONO.get(c, 0) for c in (cols or FRAC)]
    return HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=300, min_samples_leaf=30,
                                         monotonic_cst=cst, random_state=0)

def evaluate(model_fn, cols, spatial=True, k=5):
    X, y = df[cols].values, df.LST_C.values
    splitter = GroupKFold(n_splits=k) if spatial else KFold(n_splits=k, shuffle=True, random_state=0)
    pred = np.zeros(len(df))
    for tr, te in splitter.split(X, y, groups=df.block if spatial else None):
        pred[te] = model_fn(cols).fit(X[tr], y[tr]).predict(X[te])
    return dict(R2=r2_score(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)), MAE=mean_absolute_error(y, pred))

rows = []
for name, fn, cols in [('Ridge, land cover only', ridge, FRAC), ('GBoost, land cover only', gboost, FRAC),
                       ('GBoost, land cover + indices', gboost, FRAC + IDX)]:
    for spatial in (True, False):
        rows.append({'model': name, 'CV': 'spatial' if spatial else 'random', **evaluate(fn, cols, spatial)})
results = pd.DataFrame(rows).round(3)
results

**Why monotonic constraints:** your first model predicted slightly warmer temperatures when trees replaced built area. The constraints force the model to respect basic physics, which is the right fix when data is noisy. Check that R² does not drop sharply.

**Reading the table:** report the *spatial* rows. Random CV is higher because neighbouring pixels leak into training; showing both in your report demonstrates you understand why spatial CV matters. The land-cover-only model is the one used for what-if scenarios (indices come from imagery, so you cannot change them in a plan).

In [ ]:
import joblib, json, os
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1).split(df, groups=df.block))
probe = gboost(FRAC).fit(df.loc[tr, FRAC].values, df.LST_C[tr].values)
imp = permutation_importance(probe, df.loc[te, FRAC].values, df.LST_C[te].values, n_repeats=10, random_state=0)
print(pd.Series(imp.importances_mean, index=FRAC).sort_values(ascending=False).round(3))

model = gboost(FRAC).fit(df[FRAC].values, df.LST_C.values)   # final model on all data (land cover only)
os.makedirs('/content/drive/MyDrive/resilicity', exist_ok=True)
joblib.dump({'model': model, 'features': FRAC}, '/content/drive/MyDrive/resilicity/heat_model.joblib')
sp = results[(results.CV == 'spatial') & (results.model == 'GBoost, land cover only')].iloc[0]
import sklearn
json.dump({'spatial_cv': {k: float(sp[k]) for k in ['R2', 'RMSE', 'MAE']}, 'n_samples': int(len(df)),
           'range': {c: [float(df[c].min()), float(df[c].max())] for c in FRAC},   # lets the API flag out-of-range inputs
           'sklearn_version': sklearn.__version__,
           'target': 'Landsat 8/9 LST, Mar-May 2021-2025 median, deg C'},
          open('/content/drive/MyDrive/resilicity/model_card.json', 'w'), indent=2)
print('saved')

In [ ]:
APP_TO_WC = {'road': 'built', 'roof': 'built', 'pavement': 'built', 'wall': 'built', 'water': 'water'}

def to_fractions(app_percent, tree_share_of_vegetation=0.5):
    # app_percent: e.g. {'road': 34, 'wall': 26, 'roof': 18, 'vegetation': 16, 'pavement': 6}
    f = {c: 0.0 for c in FRAC}
    for k, pct in app_percent.items():
        if k == 'vegetation':
            f['f_tree'] += pct / 100 * tree_share_of_vegetation
            f['f_grass'] += pct / 100 * (1 - tree_share_of_vegetation)
        else:
            f['f_' + APP_TO_WC[k]] += pct / 100
    s = sum(f.values()); return {k: v / s for k, v in f.items()}

def scenario(before, shifts):
    # shifts e.g. {'f_built': -0.10, 'f_tree': +0.10}  = replace 10% of built area with trees
    after = {k: max(0.0, before[k] + shifts.get(k, 0.0)) for k in FRAC}
    s = sum(after.values()); after = {k: v / s for k, v in after.items()}
    p0 = float(model.predict(pd.DataFrame([before])[FRAC].values)[0]); p1 = float(model.predict(pd.DataFrame([after])[FRAC].values)[0])
    return p0, p1, p1 - p0

before = to_fractions({'road': 34, 'wall': 26, 'roof': 18, 'vegetation': 16, 'pavement': 6})
print('predicted LST before / after / change:', [round(x, 1) for x in scenario(before, {'f_built': -0.10, 'f_tree': +0.10})])

**Limits to state in your report:**
- LST is *surface* temperature, from a seasonal median at ~10:30 am local overpass, not air temperature or afternoon peak.
- The model learns correlation across the city. A predicted change from swapping land cover is a scenario estimate, not a measured effect.
- WorldCover has one `built` class, so this model cannot see cool-roof or cool-pavement albedo changes. Keep literature coefficients for those and use this model for greening and tree-canopy scenarios.
- Tree-based models are flat outside the range they saw in training. A dense street (80%+ built) may sit at the edge of your data, so check that scenario inputs fall inside the training range, and compare the Ridge model's answer too.
- Add a sanity check: compare against a second year or another season before trusting it.

---
## Part B: fine-tune the segmenter (needs your labelled photos)
Label 80-150 street photos (CVAT or Roboflow, export as PNG masks where each pixel is a class id). Put images in `data/images/` and masks in `data/masks/` with matching file names. This cell is a starting point and has not been run against your data.

In [ ]:
# !pip -q install transformers datasets evaluate accelerate
import os, glob, numpy as np, torch
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor, TrainingArguments, Trainer

LABELS = ['other', 'road', 'roof', 'pavement', 'wall', 'vegetation', 'water', 'sky']
id2label = dict(enumerate(LABELS)); label2id = {v: k for k, v in id2label.items()}
proc = SegformerImageProcessor(do_reduce_labels=False, size={'height': 512, 'width': 512})

class SegData(torch.utils.data.Dataset):
    def __init__(self, files): self.files = files
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        msk = Image.open(self.files[i].replace('images', 'masks').rsplit('.', 1)[0] + '.png')
        enc = proc(img, msk, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

files = sorted(glob.glob('data/images/*.jpg') + glob.glob('data/images/*.png'))
rng = np.random.default_rng(0); rng.shuffle(files)
n_val = max(1, len(files) // 5)
train_ds, val_ds = SegData(files[n_val:]), SegData(files[:n_val])

model = SegformerForSemanticSegmentation.from_pretrained('nvidia/mit-b0', num_labels=len(LABELS),
                                                         id2label=id2label, label2id=label2id)
args = TrainingArguments('segformer-local', learning_rate=6e-5, num_train_epochs=30,
                         per_device_train_batch_size=4, eval_strategy='epoch', save_strategy='no',
                         remove_unused_columns=False, report_to='none', fp16=torch.cuda.is_available())
Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds).train()
model.save_pretrained('segformer-local-final'); proc.save_pretrained('segformer-local-final')